# Projet : Identification de Vidéos Floutées via l'Analyse des Sous-titres

**Réalisé par : Chloé AUBINAIS, Lucas PERROT, Tim QUEFFURUS**

**Dans le cadre du cours de Gestion des Données Personnelles, Hiver 2026**

Ce notebook présente une méthode d'identification de contenu vidéo basée sur le rythme et la structure des sous-titres. Le principe est de comparer les métadonnées visuelles extraites par une IA avec une base de données de référence (fichiers SRT).

## 1. Préparation de l'environnement et des données
### Les assets
Les différents assets fournis (vidéos, sous-titres, modèle d'IA) sont à déposer dans le dossier sample_data de Google Colab. Il faut déposer directement les fichiers à la racine du dossier sample_data, et ne surtout pas les inclure dans un dossier intermédiaire.
ATTENTION : Les fichiers capture_analyse_KamouloxEp1 et capture_analyse_KamouloxEp2 doivent être placés AVANT le dossier sample_data, directement à la racine de Colab.

### Installation des bibliothèques nécessaires

Nous installons les outils pour le traitement vidéo (OpenCV), la gestion des modèles de vision (Ultralytics pour YOLO) et l'accès au SDK d'inférence.

In [ ]:
# Installation des dépendances
!pip install -q inference-sdk opencv-python ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00


### Contre-mesure d'uniformisation des sous-titres
Avec le script ci-dessous, on crée un nouveau fichier de sous-titres '.ass' où l'on impose la largeur du rectangle noir, ainsi que la taille du texte. De cette manière, la vidéo ne devrait plus être reconnue lors de l'analyse, dans le cadre de l'attaque.

In [ ]:
import os
import glob

# ==========================================
#  RÉGLAGES DU RENDU (TAILLE ET POSITION)
# ==========================================
TAILLE_TEXTE = 22       # Taille de la police
MARGE_BAS_BOITE = 15    # Élévation de la bande noire (15px du bord bas)
MARGE_BAS_TEXTE = 33    # Élévation du texte (pour le centrer dans la bande)

HAUT_BOITE = 15         # Épaisseur vers le haut
BAS_BOITE = 10          # Épaisseur vers le bas
# ==========================================

ASS_HEADER = f"""[Script Info]
Title: Full Width Permanent Banner
ScriptType: v4.00+
WrapStyle: 2
ScaledBorderAndShadow: yes

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: BgBox,Arial,20,&H60000000,&H00000000,&H00000000,&H00000000,0,0,0,0,100,100,0,0,0,0,0,1,0,0,{MARGE_BAS_BOITE},1
Style: SubText,Arial,{TAILLE_TEXTE},&H00FFFFFF,&H000000FF,&H00000000,&H00000000,0,0,0,0,100,100,0,0,1,0,0,2,0,0,{MARGE_BAS_TEXTE},1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""

def parse_srt_time_ms(time_str):
    parts = time_str.strip().replace(',', ':').split(':')
    if len(parts) == 4:
        h, m, s, ms = map(int, parts)
        return h * 3600000 + m * 60000 + s * 1000 + ms
    return 0

def format_ms_to_ass(ms):
    ms = int(max(0, ms))
    h = ms // 3600000
    ms %= 3600000
    m = ms // 60000
    ms %= 60000
    s = ms // 1000
    ms %= 1000
    cs = ms // 10
    return f"{h}:{m:02d}:{s:02d}.{cs:02d}"

def convert_srt_to_ass_permanent(input_filepath):
    if not os.path.exists(input_filepath):
        print(f"Erreur : '{input_filepath}' introuvable.")
        return

    nom_base, _ = os.path.splitext(input_filepath)
    output_filepath = f"{nom_base}.ass"

    with open(input_filepath, 'r', encoding='utf-8') as infile:
        content = infile.read()

    blocks = content.strip().split('\n\n')

    parsed_blocks = []
    for block in blocks:
        lines = block.split('\n')
        if len(lines) >= 3 and '-->' in lines[1]:
            timestamp = lines[1]
            text = '\n'.join(lines[2:])

            start_str, end_str = timestamp.split('-->')
            parsed_blocks.append({
                'start': parse_srt_time_ms(start_str),
                'end': parse_srt_time_ms(end_str),
                'text': text
            })

    parsed_blocks.sort(key=lambda x: x['start'])

    vector_shape = f"m 0 -{HAUT_BOITE} l 10000 -{HAUT_BOITE} l 10000 {BAS_BOITE} l 0 {BAS_BOITE}"

    with open(output_filepath, 'w', encoding='utf-8') as outfile:
        outfile.write(ASS_HEADER)

        rect_draw = f"{{\\p1}}{vector_shape}{{\\p0}}"
        outfile.write(f"Dialogue: 0,0:00:00.00,9:59:59.99,BgBox,,0,0,0,,{rect_draw}\n")

        for b in parsed_blocks:
            start_ass = format_ms_to_ass(b['start'])
            end_ass = format_ms_to_ass(b['end'])

            text = b['text'].replace('\n', ' ').strip()
            if text:
                outfile.write(f"Dialogue: 1,{start_ass},{end_ass},SubText,,0,0,0,,{text}\n")

    print(f"✅ Converti (Permanent) : {os.path.basename(output_filepath)}")


if __name__ == "__main__":
    dossier_cible = "/content/sample_data"
    print(f"📂 Scan du dossier '{dossier_cible}'...\n")

    fichiers_srt = glob.glob(os.path.join(dossier_cible, "*.srt"))

    if not fichiers_srt:
        print("❌ Aucun fichier .srt trouvé.")
    else:
        for filepath in fichiers_srt:
            convert_srt_to_ass_permanent(filepath)

        print(f"\n🚀 Terminé ! La bande noire part du bord GAUCHE ABSOLU vers la droite, et reste affichée en PERMANENCE.")

📂 Scan du dossier '/content/sample_data'...

✅ Converti (Permanent) : Kamoulox Ep1.ass
✅ Converti (Permanent) : Kamoulox Ep3.ass
✅ Converti (Permanent) : Kamoulox Ep4.ass
✅ Converti (Permanent) : Kamoulox Ep2.ass

🚀 Terminé ! La bande noire part du bord GAUCHE ABSOLU vers la droite, et reste affichée en PERMANENCE.


### Fusion des sous-titres et floutage

Cette étape simule le matériel de test : on incruste les sous-titres (.ass pour la vidéo que l'on défend, et .srt pour les autres) dans la vidéo, puis on applique un flou gaussien pour rendre l'image méconnaissable, ne laissant que la "forme" et le "rythme" des blocs de texte.

> Il est possible de ne traiter que les vidéos de votre choix en commentant les lignes que vous ne souhaitez pas exécuter !

La commande ci-dessous permet de générer les sous-titres selon la contre-mesure défensive, à partir du nouveau fichier de sous-titres '.ass'. Avec cette commande, la vidéo devrait ne pas être reconnue plus tard.

In [ ]:
!ffmpeg -i "sample_data/Kamoulox Ep1.mp4" -vf "ass='sample_data/Kamoulox Ep1.ass'" "sample_data/Kamoulox Ep1[MERGED].mp4" -y
#!ffmpeg -i "sample_data/Kamoulox Ep2.mp4" -vf "ass='sample_data/Kamoulox Ep2.ass'" "sample_data/Kamoulox Ep2[MERGED].mp4" -y
#!ffmpeg -i "sample_data/Kamoulox Ep3.mp4" -vf "ass='sample_data/Kamoulox Ep3.ass'" "sample_data/Kamoulox Ep3[MERGED].mp4" -y
#!ffmpeg -i "sample_data/Kamoulox Ep4.mp4" -vf "ass='sample_data/Kamoulox Ep4.ass'" "sample_data/Kamoulox Ep4[MERGED].mp4" -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

La commande ci-dessous permet d'apposer les sous-titres en respectant le fichier d'origine '.srt'. La vidéo devrait être reconnue de cette manière.

In [ ]:
#!ffmpeg -i "sample_data/Kamoulox Ep1.mp4" -vf "subtitles=sample_data/Kamoulox Ep1.srt:force_style='BackColour=&H90000000,BorderStyle=4'" "sample_data/Kamoulox Ep1[MERGED].mp4" -y
!ffmpeg -i "sample_data/Kamoulox Ep2.mp4" -vf "subtitles=sample_data/Kamoulox Ep2.srt:force_style='BackColour=&H90000000,BorderStyle=4'" "sample_data/Kamoulox Ep2[MERGED].mp4" -y
#!ffmpeg -i "sample_data/Kamoulox Ep3.mp4" -vf "subtitles=sample_data/Kamoulox Ep3.srt:force_style='BackColour=&H90000000,BorderStyle=4'" "sample_data/Kamoulox Ep3[MERGED].mp4" -y
#!ffmpeg -i "sample_data/Kamoulox Ep4.mp4" -vf "subtitles=sample_data/Kamoulox Ep4.srt:force_style='BackColour=&H90000000,BorderStyle=4'" "sample_data/Kamoulox Ep4[MERGED].mp4" -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Nous pouvons maintenant flouter les vidéos que nous venons de créer.

In [ ]:
import cv2
import os

processed_video_files = []

video_files = [
    "/content/sample_data/Kamoulox Ep1[MERGED].mp4",
    "/content/sample_data/Kamoulox Ep2[MERGED].mp4",
    #"/content/sample_data/Kamoulox Ep3[MERGED].mp4",
    #"/content/sample_data/Kamoulox Ep4[MERGED].mp4"
]

output_dir = "/content/sample_data/processed_videos"
os.makedirs(output_dir, exist_ok=True)

print("Starting video processing...")

for video_file in video_files:
    print(f"Processing video: {video_file}")

    cap = cv2.VideoCapture(video_file)

    if not cap.isOpened():
        print(f"Error: Could not open video file {video_file}. Skipping.")
        continue

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec pour MP4

    base_name = os.path.basename(video_file)
    name_without_ext, ext = os.path.splitext(base_name)
    output_filename = f"{name_without_ext}_blurred{ext}"
    output_filepath = os.path.join(output_dir, output_filename)

    out = cv2.VideoWriter(output_filepath, fourcc, fps, (frame_width, frame_height))

    if not out.isOpened():
        print(f"Error: Could not open video writer for {output_filepath}. Skipping.")
        cap.release()
        continue

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        blurred_frame = cv2.GaussianBlur(frame, (21, 21), 0)

        out.write(blurred_frame)

    cap.release()
    out.release()

    processed_video_files.append(output_filepath)
    print(f"Finished processing and saved to: {output_filepath}")

print("\nAll videos processed.")
print("Processed video files:")
for processed_file in processed_video_files:
    print(processed_file)

Starting video processing...
Processing video: /content/sample_data/Kamoulox Ep1[MERGED].mp4
Finished processing and saved to: /content/sample_data/processed_videos/Kamoulox Ep1[MERGED]_blurred.mp4
Processing video: /content/sample_data/Kamoulox Ep2[MERGED].mp4
Finished processing and saved to: /content/sample_data/processed_videos/Kamoulox Ep2[MERGED]_blurred.mp4

All videos processed.
Processed video files:
/content/sample_data/processed_videos/Kamoulox Ep1[MERGED]_blurred.mp4
/content/sample_data/processed_videos/Kamoulox Ep2[MERGED]_blurred.mp4


## 2. Création de la Base de Données de Référence

Nous transformons les fichiers .srt physiques en un fichier JSON structuré. Ce fichier sert de "dictionnaire" pour l'algorithme de comparaison.

In [2]:
import os
import json
import re

def parse_srt_time(time_str):
    """Convertit le format de temps SRT en secondes."""
    time_str = time_str.replace(',', '.')
    h, m, s = re.split(':', time_str)
    return int(h) * 3600 + int(m) * 60 + float(s)

def parse_srt_file(file_path):
    """Lit un fichier .srt physique et extrait les données temporelles et spatiales."""
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()

    blocks = content.split('\n\n')
    subtitles = []

    for block in blocks:
        lines = block.split('\n')
        if len(lines) >= 3:
            try:
                time_line = lines[1]
                text_lines = lines[2:]
                start_str, end_str = time_line.split(' --> ')

                subtitles.append({
                    'start': parse_srt_time(start_str),
                    'end': parse_srt_time(end_str),
                    'lines_count': len(text_lines)
                })
            except Exception as e:
                # Ignorer les blocs mal formatés
                continue

    return subtitles

def build_database(srt_directory, output_json="database.json"):
    """Parcourt le dossier et crée la base de données JSON."""
    database = {}

    # Parcourir tous les fichiers SRT du dossier
    for filename in os.listdir(srt_directory):
        if filename.endswith(".srt"):
            file_path = os.path.join(srt_directory, filename)
            video_name = filename.replace('.srt', '')

            print(f"Traitement de : {filename}...")
            parsed_data = parse_srt_file(file_path)
            database[video_name] = parsed_data

    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(database, f, indent=4)

    print(f"\n✅ Base de données créée avec succès ! {len(database)} vidéos stockées dans {output_json}.")

# ==========================================
# EXÉCUTION
# ==========================================
if __name__ == "__main__":
    # Dossier où sont les fichiers SRT
    folder_path = "sample_data"

    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Dossier '{folder_path}' créé. Y mettre les fichiers .srt et relancer le script.")
    else:
        build_database(folder_path)

Traitement de : Kamoulox Ep4.srt...
Traitement de : Kamoulox Ep2.srt...
Traitement de : Kamoulox Ep1.srt...
Traitement de : Kamoulox Ep3.srt...

✅ Base de données créée avec succès ! 4 vidéos stockées dans database.json.


## 3. Analyse de la Vidéo par Intelligence Artificielle

Ici, nous utilisons un modèle YOLO (localement) pour détecter les boîtes englobantes des sous-titres dans la vidéo floutée. Une machine à états analyse si le sous-titre change en fonction de sa surface et de sa position.

> Vous pouvez passer cette étape et utiliser les fichiers d'analyse fournis pour l'étape 4.

In [ ]:
import cv2
import numpy as np
import json
from ultralytics import YOLO

# Localisation du modèle IA
model = YOLO("/content/sample_data/best.pt")

video_path = "/content/sample_data/processed_videos/Kamoulox Ep2[MERGED]_blurred.mp4"
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_duration_T = total_frames / fps if fps > 0 else 0

width_video = cap.get(cv2.CAP_PROP_FRAME_WIDTH)

subtitles_sequence = []
active_subtitle = None
frames_since_last_sub = 0
frame_count = 0

print(f"L'IA analyse la vidéo (Durée totale : {video_duration_T:.2f}s)... Cette opération est longue ! (env. 10mn pour une vidéo de 2mn)")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    current_time = frame_count / fps
    frame_count += 1

    results = model(frame, conf=0.50, verbose=False)

    valid_box = None

    num_predictions = len(results[0].boxes)
    #print(f'{frame_count} | {num_predictions}')

    if num_predictions > 0:
        boxes = results[0].boxes

        best_idx = boxes.conf.argmax().item()
        best_box = boxes[best_idx]

        x_center, y_center, w, h = best_box.xywh[0].cpu().numpy()

        valid_box = {
            'height': float(h),
            'width': float(w),
            'x': float(x_center),
            'area': float(w * h)
        }

    # --- MACHINE À ÉTATS ---
    if valid_box is not None:
        if active_subtitle is None:
            active_subtitle = {
                'start': current_time,
                'end': current_time,
                'heights': [valid_box['height']],
                'widths': [valid_box['width']],
                'xs': [valid_box['x']],
                'areas': [valid_box['area']]
            }
            frames_since_last_sub = 0
        else:
            # On lisse les valeurs avec la médiane
            median_area = np.median(active_subtitle['areas'])
            median_x = np.median(active_subtitle['xs'])
            median_h = np.median(active_subtitle['heights'])

            text_changed = False

            # Critère 1 : La surface change de plus de 12%
            if abs(valid_box['area'] - median_area) > (median_area * 0.12):
                text_changed = True
                print("Critère 1")

            # Critère 2 : Le point X se décale significativement
            elif abs(valid_box['x'] - median_x) > 10:
                text_changed = True
                print("Critère 2")

            if text_changed:
                print("NOUVEAU SOUS-TITRE")
                # COUPURE : On enregistre le sous-titre terminé
                quantified_height = round(median_h / 10) * 10
                subtitles_sequence.append({
                    'start': round(active_subtitle['start'], 3),
                    'end': round(active_subtitle['end'], 3),
                    'lines_count': quantified_height
                })

                # DÉMARRAGE : On lance le nouveau sous-titre
                active_subtitle = {
                    'start': current_time,
                    'end': current_time,
                    'heights': [valid_box['height']],
                    'widths': [valid_box['width']],
                    'xs': [valid_box['x']],
                    'areas': [valid_box['area']]
                }
            else:
                # PROLONGATION : C'est le même sous-titre
                active_subtitle['end'] = current_time
                active_subtitle['heights'].append(valid_box['height'])
                active_subtitle['widths'].append(valid_box['width'])
                active_subtitle['xs'].append(valid_box['x'])
                active_subtitle['areas'].append(valid_box['area'])

            frames_since_last_sub = 0

    else:
        # Gestion des blancs entre les sous-titres
        if active_subtitle is not None:
            frames_since_last_sub += 1
            if frames_since_last_sub > 2: # Tolérance de 2 frames
                median_h = np.median(active_subtitle['heights'])
                quantified_height = round(median_h / 10) * 10
                subtitles_sequence.append({
                    'start': round(active_subtitle['start'], 3),
                    'end': round(active_subtitle['end'], 3),
                    'lines_count': quantified_height
                })
                active_subtitle = None

cap.release()

if active_subtitle is not None:
    median_h = np.median(active_subtitle['heights'])
    subtitles_sequence.append({
        'start': round(active_subtitle['start'], 3),
        'end': round(active_subtitle['end'], 3),
        'lines_count': (round(median_h / 10) * 10)/20
    })

print(f"\nExtraction terminée ! {len(subtitles_sequence)} sous-titres détectés.")

# ==========================================
# --- CRÉATION ET SAUVEGARDE DU JSON ---
# ==========================================

output_data = {
    "duree_totale_T": round(video_duration_T, 3),
    "nombre_silhouettes_m": len(subtitles_sequence),
    "sequence": subtitles_sequence
}

output_filename = "capture_analyse.json"
with open(output_filename, 'w', encoding='utf-8') as json_file:
    json.dump(output_data, json_file, indent=4)

print(f"✅ Analyse sauvegardée avec succès dans le fichier : '{output_filename}' !")

L'IA analyse la vidéo (Durée totale : 133.92s)... Cette opération est longue ! (env. 10mn pour une vidéo de 2mn)
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU SOUS-TITRE
Critère 1
NOUVEAU

## 4. Identification Algorithmique (Prédiction)

Enfin, les séquences de sous-titres sont converties en chronologies binaires continues (présence ou absence de texte). L'algorithme utilise une fenêtre glissante pour calculer la Corrélation de Pearson entre la capture et la base de données, évaluant ainsi l'alignement structurel des dialogues et des silences. Ce résultat est ensuite affiné par un filtre pénalisant les détections intrusives (faux positifs lors des silences absolus), ce qui permet d'établir un score de confiance final extrêmement robuste.

> Pour gagner du temps, si vous avez passé l'analyse par IA, vous pouvez commenter l'avant-dernière ligne et dé-commenter la dernière ligne du code ci-dessous, et modifier le nom du fichier pour utiliser les exemples fournis capture_analyse_KamouloxEp1.json et capture_analyse_KamouloxEp2.json.

In [4]:
import json
import numpy as np

# ==========================================
# 1. MOTEUR CHRONOLOGIQUE ET CORRÉLATION
# ==========================================

def build_timeline(sequence, duration_T=None, resolution=10):
    """
    Transforme la liste en un tableau binaire (10 cases par seconde).
    """
    if not sequence: return np.array([])
    if duration_T is None:
        duration_T = sequence[-1]['end']

    length = int(duration_T * resolution) + 1
    timeline = np.zeros(length, dtype=np.int8)

    for sub in sequence:
        start_idx = max(0, int(sub['start'] * resolution))
        end_idx = min(length, int(sub['end'] * resolution))
        timeline[start_idx:end_idx] = 1

    return timeline

def find_best_alignment(cap_timeline, db_timeline):
    """
    Fait glisser la capture sur toute la base de données pour trouver
    la meilleure Corrélation de Pearson, puis applique un filtre
    impitoyable sur l'intégrité des silences.
    """
    cap_len = len(cap_timeline)
    db_len = len(db_timeline)

    if cap_len > db_len:
        return 0.0, 0

    # On centre la capture sur 0 pour éliminer le biais du "bavardage"
    cap_mean = np.mean(cap_timeline)
    if cap_mean == 1.0 or cap_mean == 0.0:
        return 0.0, 0 # Pas de variance (100% de texte ou 100% de vide)

    cap_norm = cap_timeline - cap_mean
    cap_std_sum = np.sum(cap_norm**2)

    best_score = -1.0
    best_offset = 0

    # On fait glisser la fenêtre (Convolution)
    for i in range(db_len - cap_len + 1):
        db_window = db_timeline[i : i + cap_len]
        db_mean = np.mean(db_window)

        if db_mean == 1.0 or db_mean == 0.0:
            continue

        db_norm = db_window - db_mean

        # Formule de Pearson
        numerator = np.sum(cap_norm * db_norm)
        denominator = np.sqrt(cap_std_sum * np.sum(db_norm**2))

        if denominator == 0: continue

        score = numerator / denominator

        if score > best_score:
            best_score = score
            best_offset = i

    # =========================================================
    # LE FILTRE DE VIOLATION DE SILENCE
    # =========================================================
    if best_score > 0:
        # On extrait le segment exact de la base de données qui a "matché"
        best_db_window = db_timeline[best_offset : best_offset + cap_len]

        # Combien de "cases temps" sont censées être du silence absolu ?
        silences_in_db = np.sum(best_db_window == 0)

        if silences_in_db > 0:
            # Combien de fois l'IA a-t-elle détecté du texte (1) pendant ces silences (0) ?
            violations = np.sum((best_db_window == 0) & (cap_timeline == 1))

            # Calcul du ratio de pollution (0.0 = parfait, 1.0 = tous les silences sont pollués)
            violation_ratio = violations / silences_in_db

            # Le Malus : On retire directement des points de pourcentage en fonction de la pollution.
            malus = violation_ratio * 100

            final_score = (best_score * 100) - malus
        else:
            final_score = best_score * 100
    else:
        final_score = 0.0

    # On s'assure que le score ne tombe pas en dessous de 0
    return max(0.0, final_score), best_offset


# ==========================================
# 2. ALGORITHME DE RECHERCHE
# ==========================================

def identify_video(capture_file="capture_analyse.json", db_file="database.json"):
    try:
        with open(capture_file, 'r', encoding='utf-8') as f:
            capture_data = json.load(f)
        with open(db_file, 'r', encoding='utf-8') as f:
            database = json.load(f)
    except FileNotFoundError as e:
        print(f"❌ Erreur : Fichier introuvable ({e.filename}).")
        return

    T_capture = capture_data['duree_totale_T']
    cap_sequence = capture_data['sequence']

    if not cap_sequence:
        print("❌ La capture est vide.")
        return

    print("-" * 50)
    print("Recherche avec Corrélation de Pearson (Numpy)...")
    print(f"Durée de la capture : {T_capture}s")
    print("-" * 50)

    # Création du vecteur de l'enregistrement
    cap_timeline = build_timeline(cap_sequence, T_capture)
    candidates = []

    for video_name, db_sequence in database.items():
        if not db_sequence: continue

        db_duration = db_sequence[-1]['end']

        # Sécurité : Si le morceau de DB fourni est plus court que la capture, on l'allonge avec du vide
        if db_duration < T_capture:
            db_timeline = build_timeline(db_sequence, T_capture)
        else:
            db_timeline = build_timeline(db_sequence, db_duration)

        score, offset_idx = find_best_alignment(cap_timeline, db_timeline)

        # Un score > 30% après malus reste une correspondance valide pour une vidéo normale
        if score > 30.0:
            start_time_seconds = offset_idx / 10.0 # On reconvertit l'index en secondes
            candidates.append({
                'video_name': video_name,
                'start_time': start_time_seconds,
                'confidence': round(score, 2)
            })

    if not candidates:
        print("❌ Aucune correspondance trouvée.")
        return

    # Tri par confiance
    candidates.sort(key=lambda x: x['confidence'], reverse=True)

    print(f"Correspondance trouvée !")
    print("\nTOP RÉSULTATS :")

    for i, res in enumerate(candidates[:5]):
        start_fmt = f"{int(res['start_time']//3600):02d}:{int((res['start_time']%3600)//60):02d}:{int(res['start_time']%60):02d}"

        print(f"\nVidéo                : {res['video_name']}")
        print(f"Score de Corrélation : {res['confidence']}%")
        print(f"Début estimé         : {start_fmt}")

if __name__ == "__main__":
    identify_video()
    #identify_video("capture_analyse_KamouloxEp2.json")

--------------------------------------------------
Recherche avec Corrélation de Pearson (Numpy)...
Durée de la capture : 133.92s
--------------------------------------------------
Correspondance trouvée !

TOP RÉSULTATS :

Vidéo                : Kamoulox Ep2
Score de Corrélation : 64.24%
Début estimé         : 00:00:00
